# agentic-rl-wordle — 多輪 GRPO 訓練（Colab A100）

流程：①參數 → ②Drive+bundle+安裝 → ③HF_TOKEN → ④pytest 煙霧 → ⑤(選)spike → ⑥啟動訓練 → ⑦監控 → ⑧收尾（評測+push+釋放）。

- **SMOKE_TEST=True** 先跑猜數字煙霧（~30 分鐘內要看到 reward/win_rate 上升，M2.3 gate）
- 斷線 SOP：重新執行 ①②③⑥（RESUME="auto" 會從 Drive 最新 checkpoint 接續）
- 背景執行請開：執行階段 → 背景執行；**push 完會自動 runtime.unassign() 釋放機器**

In [ ]:
# ===== ① 參數（唯一需要手改的 cell）=====
SMOKE_TEST = True                 # True: 猜數字煙霧；False: Wordle 正式訓練
RUN_NAME = "wordle-grpo-v1"       # Drive 上 checkpoint/log 的資料夾名
HF_USERNAME = "steven0226"
REWARD_PRESET = "shaped"          # shaped | binary（HF 官方發現 binary 對 Wordle 更穩，留作 A/B）
MAX_HOURS = 8.0                   # 正式訓練牆鐘預算，到點自動存檔停訓
RESUME = "auto"

DRIVE_BASE = "/content/drive/MyDrive/agentic-rl-wordle"

In [ ]:
# ===== ② Drive + 原始碼 bundle + 依賴安裝 =====
from google.colab import drive
drive.mount('/content/drive')

import pathlib
BUNDLE = f"{DRIVE_BASE}/wordle_rl_bundle.zip"
assert pathlib.Path(BUNDLE).exists(), f"先在本機執行 scripts/make_colab_bundle.py 並把 zip 上傳到 {BUNDLE}"

!rm -rf /content/agentic-rl-wordle && mkdir -p /content/agentic-rl-wordle
!unzip -q -o "$BUNDLE" -d /content/agentic-rl-wordle
%cd /content/agentic-rl-wordle

# ⚠️ 版本 pin 原則：torch 用 Colab 內建 CUDA build（絕不覆蓋，專案 2 教訓）；
#    trl/vllm/peft 精確 pin 於 requirements-colab.txt（M2.1 spike 已驗證過此組合可行）
!pip install -q -e . pytest
!pip install -q -r requirements-colab.txt

import torch, transformers, trl, vllm
print("torch", torch.__version__, "| trl", trl.__version__,
      "| transformers", transformers.__version__, "| vllm", vllm.__version__)
assert trl.__version__.startswith("1.8"), "trl 版本漂移——對照 requirements-colab.txt 與 docs/decision.md"

# 單字表 fetch-at-setup（出處與 sha256 → data/SOURCE.json；計數斷言 2315/10657/12972）
!python scripts/fetch_words.py

In [ ]:
# ===== ③ HF_TOKEN（Colab Secrets 需事先設定）=====
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN OK")

In [ ]:
# ===== ④ 煙霧驗證（<60 秒；不全綠就不要燒 GPU）=====
!python -m pytest tests -q

In [ ]:
# ===== ⑤（選）M2.1 spike：新環境首跑必執行——驗證 TRL 接觸面、記錄版本三元組 =====
# 過 → 把印出的版本回填 requirements-colab.txt、結論寫 docs/decision.md
# 不過 → 依 SpikeValidationError 訊息修 rollout.py / train.py；60 分鐘 timebox，超時轉 ART 備援
if SMOKE_TEST:
    !python scripts/spike_trl.py

In [ ]:
# ===== ⑥ 啟動訓練（subprocess + log 落 Drive；cell 直跑會被 websocket 拖死）=====
import pathlib, subprocess, sys
CKPT_DIR = f"{DRIVE_BASE}/runs/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else "")
pathlib.Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
LOG_PATH = f"{CKPT_DIR}/train.log"

cmd = [sys.executable, "-m", "wordle_rl.train",
       "--preset", "smoke" if SMOKE_TEST else "full",
       "--reward", REWARD_PRESET,
       "--output-dir", CKPT_DIR,
       "--resume", RESUME,
       "--max-hours", str(0.5 if SMOKE_TEST else MAX_HOURS)]
print(" ".join(cmd))
log_f = open(LOG_PATH, "ab")
proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
print("PID:", proc.pid, "| log:", LOG_PATH)

In [ ]:
# ===== ⑦ 監控（可重複執行；曲線讀 metrics.jsonl）=====
import json, pathlib
print("--- train.log 尾端 ---")
log_text = pathlib.Path(LOG_PATH).read_text(errors="ignore")
print("\n".join(log_text.splitlines()[-25:]))

mpath = pathlib.Path(CKPT_DIR) / "metrics.jsonl"
if mpath.exists():
    rows = [json.loads(l) for l in mpath.read_text().splitlines() if l.strip()]
    if rows:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
        for ax, key, title in [
            (axes[0], "reward/mean", "mean episode reward"),
            (axes[1], "rollout/win_rate", "rollout win rate"),
            (axes[2], "rollout/illegal_per_ep", "illegal turns / episode"),
        ]:
            pts = [(r["step"], r[key]) for r in rows if r.get(key) is not None]
            if pts:
                ax.plot(*zip(*pts))
            ax.set_title(title); ax.set_xlabel("step")
        plt.tight_layout(); plt.show()
        keep = ("step", "reward/mean", "rollout/win_rate", "reward/zero_variance_group_frac")
        print("最新：", {k: v for k, v in rows[-1].items() if k in keep})
else:
    print("metrics.jsonl 尚未出現（暖身中）")
print("\n樣本 transcript：", sorted(pathlib.Path(CKPT_DIR, "samples").glob("*.md"))[-3:] if pathlib.Path(CKPT_DIR, "samples").exists() else "尚無")

In [ ]:
# ===== ⑧ 收尾：等訓練結束 → 前後對照評測 → push HF → 釋放機器 =====
# ⚠️ 會阻塞直到訓練結束；push 失敗不會擋 runtime.unassign()（Drive 是真相來源，可事後補 push）
import shutil, subprocess, sys
try:
    rc = proc.wait()
    print("訓練結束 returncode =", rc)
    if not SMOKE_TEST:
        adapter = f"{CKPT_DIR}/final"
        subprocess.run([sys.executable, "eval/run_eval.py",
                        "--adapter", adapter, "--backend", "vllm"], check=True)
        shutil.copytree("results", f"{CKPT_DIR}/results", dirs_exist_ok=True)
        subprocess.run([sys.executable, "scripts/push_model.py",
                        "--adapter", adapter,
                        "--repo", f"{HF_USERNAME}/qwen2.5-1.5b-wordle-grpo",
                        "--card", "docs/model_card.md"], check=True)
        print("HF push 完成 ✅")
finally:
    from google.colab import runtime
    runtime.unassign()